In [24]:
import torch

doc_ids = torch.tensor([[0, 0, 0, 1, 1, 1, 1, 2, 2]]).expand(2, -1)

In [25]:
batch, seq_len = doc_ids.shape

In [26]:
# Find where groups change
is_new_group = torch.cat([
    torch.ones_like(doc_ids[:, :1], dtype=torch.bool),
    doc_ids[:, 1:] != doc_ids[:, :-1]
], dim=1)

In [27]:
torch.where(is_new_group)

(tensor([0, 0, 0, 1, 1, 1]), tensor([0, 3, 7, 0, 3, 7]))

In [28]:
# Get the indices where new groups start
group_start_indices = torch.where(is_new_group)[1]
group_start_indices = group_start_indices.view(batch, -1)

In [29]:
# Broadcast subtraction: subtract the start index of current group from position
positions = torch.arange(doc_ids.size(1)).unsqueeze(0).expand_as(doc_ids)
group_starts = torch.zeros_like(positions)

In [30]:
is_new_group

tensor([[ True, False, False,  True, False, False, False,  True, False],
        [ True, False, False,  True, False, False, False,  True, False]])

In [31]:
group_starts[:, is_new_group[0]], group_start_indices

(tensor([[0, 0, 0],
         [0, 0, 0]]),
 tensor([[0, 3, 7],
         [0, 3, 7]]))

In [32]:
group_starts[:, is_new_group[0]] = group_start_indices

In [33]:
group_starts

tensor([[0, 0, 0, 3, 0, 0, 0, 7, 0],
        [0, 0, 0, 3, 0, 0, 0, 7, 0]])

In [34]:
group_starts = group_starts.cummax(dim=1)[0]  # Forward fill the start indices

In [35]:
group_starts, positions

(tensor([[0, 0, 0, 3, 3, 3, 3, 7, 7],
         [0, 0, 0, 3, 3, 3, 3, 7, 7]]),
 tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8],
         [0, 1, 2, 3, 4, 5, 6, 7, 8]]))

In [36]:
positions = positions - group_starts

In [37]:
positions

tensor([[0, 1, 2, 0, 1, 2, 3, 0, 1],
        [0, 1, 2, 0, 1, 2, 3, 0, 1]])

In [38]:
pos_emb  =torch.randn(5, 16) # max 5 positions, embedding dim 16

In [39]:
pos_emb[positions].shape

torch.Size([2, 9, 16])